<a href="https://colab.research.google.com/github/davidekim/WRAPs/blob/main/mspa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Pipeline example for creating WRAPed MspA**

In [ ]:
#@title **Setup RFdiffusion and pyrosetta** (~5-10min)
%%time
import os, time
import sys
import subprocess

def run_cmd(cmd):
  process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=True, text=True)
  for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()
  process.stdout.close()
  process.wait()

if not os.path.isdir("RFdiffusion"):
  print("installing RFdiffusion...")
  os.system("git clone https://github.com/RosettaCommons/RFdiffusion.git")
  # install dependencies
  os.system("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
  os.system("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
  os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
  os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
  os.system("cd RFdiffusion/env/SE3Transformer; pip install .")
  os.system("pip install biopython==1.81")
  os.system("pip install -U dm-haiku")
  os.system("pip install ml-collections")
  os.system('pip install --upgrade "jax[cuda12_pip]<0.6.0" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html')
  os.system("cd RFdiffusion; pip install -e .")
  # install PyRosetta
  os.system("pip install pyrosetta --find-links https://west.rosettacommons.org/pyrosetta/quarterly/release")
  os.system("pip install py3Dmol")
print()

os.environ["DGLBACKEND"] = "pytorch"
#os.environ["HYDRA_FULL_ERROR"] = '1'
diffusion_script = "RFdiffusion/scripts/run_inference.py"

In [ ]:
#@title **Install RFdiffusion weights** (~1-5min)
%%time
if not os.path.isdir("RFdiffusion/models"):
  print("installing RFdiffusion weights...")
  # Only install what is used in this notebook
  os.system("cd RFdiffusion; mkdir models && cd models")
  #os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt")
  #os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/60f09a193fb5e5ccdc4980417708dbab/Complex_Fold_base_ckpt.pt")
  #os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/74f51cfb8b440f50d70878e05361d8f0/InpaintSeq_ckpt.pt")
  #os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/76d00716416567174cdb7ca96e208296/InpaintSeq_Fold_ckpt.pt")
  #os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/5532d2e1f3a4738decd58b19d633b3c3/ActiveSite_ckpt.pt")
  #os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/12fc204edeae5b57713c5ad7dcb97d39/Base_epoch8_ckpt.pt")

In [ ]:
#@title **Get ProteinMPNN git repo**
%%time
import os, time
if not os.path.isdir("ProteinMPNN"):
  run_cmd("git clone https://github.com/dauparas/ProteinMPNN.git")

In [ ]:
#@title **Get WRAPs git repo**
%%time
import os, time
if not os.path.isdir("WRAPs"):
  run_cmd("git clone https://github.com/davidekim/WRAPs.git")

In [ ]:
#@title **Add length and secondary structure elements for the WRAP**
#
# First, the PDB file of the target protein is provided as input. The script
# then extracts the secondary structure elements of the native membrane protein,
# where H denotes a helix, L a loop, and E a β-sheet.
# After padding for the WRAP, secondary structure elements are inserted at the N- or C-terminus.
# Elements are indexed starting from 0. For example, adding a helix–loop–helix motif to the
# N-terminus would correspond to elements 0–1–2, followed by the native target secondary structure
# (ss_types2). Finally, ss_types3 represents the complete secondary structure sequence, containing
# both the newly added WRAP elements and the native target elements arranged in the correct order
# as they should appear in the final structure.
#
# This block adjacency generation code was developed by Thomas Schlichthärle and Ljubica Mihaljevic
# specifically for MspA.

import pyrosetta
import pyrosetta.rosetta.core.import_pose as import_pr_pose
from pyrosetta import *
from pyrosetta.rosetta import *
from pyrosetta.rosetta.core import *
pyrosetta.init(" -mute all ")

pdb_name = 'WRAPs/WRAP_MspA/mspa_cut.pdb'

def ss_Type_lengths(pdb_name):
  DSSP = pyrosetta.rosetta.protocols.moves.DsspMover()
  pose = import_pr_pose.pose_from_file(pdb_name)
  DSSP.apply(pose)
  DSSP_pose_string = []
  SS_type_lengths = []
  current_SS = ''
  helix_counter = 0
  Loop_counter = 0
  Sheet_counter = 0
  for z in range(1, len(pose.sequence())):
    if current_SS == pose.secstruct(z):
      SS_type_lengths[len(SS_type_lengths)-1][1] = SS_type_lengths[len(SS_type_lengths)-1][1] +1
    if current_SS != pose.secstruct(z):
      SS_type_lengths.append([pose.secstruct(z),1])
      if pose.secstruct(z) == "H":
        helix_counter = helix_counter+1
      if pose.secstruct(z) == "L":
        Loop_counter = Loop_counter+1
      if pose.secstruct(z) == "E":
        Sheet_counter = Sheet_counter+1
      current_SS = pose.secstruct(z)
      DSSP_pose_string.append(pose.secstruct(z))
  return SS_type_lengths

ss_types = ss_Type_lengths(pdb_name)
ss_types2 = ss_types.copy()
ss_types2.insert(0,['H',60])
ss_types2.insert(1,['L',3])
ss_types2.insert(2,['H',20])
ss_types2.insert(11,['H',60])
ss_types2.insert(12,['L',3])
ss_types2.insert(13,['H',20])
ss_types2.insert(22,['H',60])
ss_types2.insert(23,['L',3])
ss_types2.insert(24,['H',20])
ss_types2.insert(33,['H',60])
ss_types2.insert(34,['L',3])
ss_types2.insert(35,['H',20])
ss_types2.insert(44,['H',60])
ss_types2.insert(45,['L',3])
ss_types2.insert(46,['H',20])
ss_types2.insert(55,['H',60])
ss_types2.insert(56,['L',3])
ss_types2.insert(57,['H',20])
ss_types2.insert(66,['H',60])
ss_types2.insert(67,['L',3])
ss_types2.insert(68,['H',20])
ss_types2.insert(77,['H',60])
ss_types2.insert(78,['L',3])
ss_types2.insert(79,['H',20])

ss_types3 = [['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9],
 ['H', 60],
 ['L', 3],
 ['H', 20],
 ['L', 2],
 ['E', 11],
 ['L', 4],
 ['E', 5],
 ['L', 11],
 ['E', 2],
 ['L', 6],
 ['E', 9]]

In [ ]:
#@title **Specify which elements interact with the target and within the WRAP**
# Secondary structure elements are guided by explicitly defining which elements are
# allowed to interact with one another. This is done by specifying interaction pairs
# between element indices. For example, element 0 can be defined to interact with
# elements 1, 2, 3, and 43 (the last element).

adjacencies = [[1, 2, 3, 43],
 [0, 4, 5, 6],
 [0, 5],
 [0],
 [1],
 [1, 2],
 [1, 7, 8, 9],
 [6, 10, 11, 12],
 [6, 11],
 [6],
 [7],
 [7, 8],
 [7, 13, 14, 15],
 [12, 16, 17, 18],
 [12, 17],
 [12],
 [13],
 [13, 14],
 [13, 19, 20, 21],
 [18, 22, 23, 24],
 [18, 23],
 [18],
 [19],
 [19, 20],
 [19, 25, 26, 27],
 [24, 28, 29, 30],
 [24, 29],
 [24],
 [25],
 [25, 26],
 [25, 31, 32, 33],
 [30, 34, 35, 36],
 [30, 35],
 [30],
 [31],
 [31, 32],
 [31, 37, 38, 39],
 [36, 40, 41, 42],
 [36, 41],
 [36],
 [37],
 [37, 38],
 [37, 43, 44, 45],
 [0, 42, 46, 47],
 [42, 47],
 [42],
 [43],
 [43, 44]]

In [ ]:
#@title **Generate block_adjacency for diffusion**
# Next, translate these specifications into a format that the diffusion
# model can understand and specify the file name and save this information
# for use during diffusion-based structure generation.

import torch
import matplotlib.pyplot as plt

adjacency_matrix_length = 1064
tensor_adj_matrix = torch.zeros(adjacency_matrix_length,adjacency_matrix_length)

# Get indices for start and ends of helices
helix_indices = []
current_start_index = 0
current_end_index = -1
for i in range(0, len(ss_types3)):
  current_end_index = current_end_index + ss_types3[i][1]
  if ss_types3[i][0] == 'H':
    helix_indices.append([current_start_index,current_end_index])
  if ss_types3[i][0] == 'E':
    helix_indices.append([current_start_index,current_end_index])
  current_start_index = current_start_index + ss_types3[i][1]

# Generate block adjacency by helices
padded_adj_matrix2=tensor_adj_matrix
current_start_indices = 0
current_end_indices = 0
helix_counter = 0

for i in range(0, len(ss_types3)):
  current_end_indices = current_end_indices + ss_types3[i][1]
  if ss_types3[i][0] == 'H':
    for k in range(0, len(adjacencies[helix_counter])):
      for l in range(helix_indices[helix_counter][0],helix_indices[helix_counter][1]+1):
        for j in range(helix_indices[adjacencies[helix_counter][k]][0],helix_indices[adjacencies[helix_counter][k]][1]+1):
          padded_adj_matrix2[l, j] = 1

    helix_counter = helix_counter+1
  if ss_types3[i][0] == 'E':
    for k in range(0, len(adjacencies[helix_counter])):
      for l in range(helix_indices[helix_counter][0],helix_indices[helix_counter][1]+1):
        for j in range(helix_indices[adjacencies[helix_counter][k]][0],helix_indices[adjacencies[helix_counter][k]][1]+1):
          padded_adj_matrix2[l, j] = 1
    helix_counter = helix_counter+1
  current_start_indices = current_start_indices + ss_types3[i][1]

tensor_ss_strand = torch.zeros(len(padded_adj_matrix2[0]))
start_position = 0
end_position = 0

for i in range(0, len(ss_types3)):
    end_position = end_position + ss_types3[i][1]
    if ss_types3[i][0] == 'H':
        tensor_ss_strand[start_position:end_position] = 0

    if ss_types3[i][0] == 'E':
        tensor_ss_strand[start_position:end_position] = 1

    if ss_types3[i][0] == 'L':
        tensor_ss_strand[start_position:end_position] = 2

    start_position = start_position + ss_types3[i][1]

if not os.path.isdir("mspa_longer"): os.mkdir("mspa_longer")
ss_strand_filepath = 'mspa_longer/mspA_ss.pt'
adj_matrix_filepath = 'mspa_longer/mspA_adj.pt'

torch.save(tensor_ss_strand, ss_strand_filepath)
torch.save(padded_adj_matrix2, adj_matrix_filepath)

plt.imshow(padded_adj_matrix2)

In [ ]:
#@title **Run scaffold-guided RFdiffusion with symmetry** (~10min to many hours)
%%time
num_designs = 5 #@param ["1","2","3","4","5","6","7","8","9","10"] {type:"raw"}

cmd = f"RFdiffusion/scripts/run_inference.py "
cmd += f"--config-name=symmetry inference.symmetry='C8' "
cmd += f"inference.output_prefix=mspa/mspa inference.input_pdb=WRAPs/WRAP_MspA/mspa_cut.pdb "
cmd += f"inference.num_designs={num_designs} "
cmd += f"contigmap.contigs=['83/A1-50/0 83/B1-50/0 83/C1-50/0 83/D1-50/0 83/E1-50/0 83/F1-50/0 83/G1-50/0 83/H1-50/0'] "
cmd += f"inference.model_runner=NRBStyleSelfCond denoiser.noise_scale_ca=0.5 denoiser.noise_scale_frame=0.5 "
cmd += f"scaffoldguided.scaffoldguided=True scaffoldguided.scaffold_dir=mspa_longer diffuser.T=30"

print(cmd)
run_cmd(cmd)

rfd_mods = []
import glob
for rfd in glob.glob('mspa/mspa*.pdb'):
  rfd_mods.append(rfd)

In [ ]:
#@title **Select scaffold-guided RFdiffusion output for MPNN and AF3**
import py3Dmol
import ipywidgets as widgets
from ipywidgets import interact, Layout
from IPython.display import display, clear_output

current_rfd = ""
dropdown = widgets.Dropdown(
  options=rfd_mods,
  description='backbones:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(rfd):
  global current_rfd
  current_rfd = rfd
  clear_output(wait=True)
  print()
  print(current_rfd)
  view = py3Dmol.view(width=500, height=400)
  with open(rfd, "r") as f:
    pdb_data = f.read()
  view.addModel(pdb_data, 'pdb')
  view.setStyle({'cartoon': {'colorscheme': 'chain'}})
  view.zoomTo()
  view.show()

widgets.interact(on_dropdown_change, rfd=dropdown);



In [ ]:
#@title **Run tied (symmetric) Soluble ProteinMPNN**
%%time
if not os.path.exists(current_rfd):
  raise Exception("You must select an RFdiffused backbone.")
from pyrosetta.rosetta.core.pose import append_pose_to_pose
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.protocols.minimization_packing import MinMover

#@markdown Select to minimize sidechains in MPNN PDB outputs (optional for aesthetics)
minimize = False #@param {type:"boolean"}
sequences_per_target = 5 #@param ["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20" ] {type: "raw"}

import glob

# copy rfd backbone to run mpnn and af3 on
designsdir = "mspa_rfd"
if not os.path.isdir(designsdir):
  os.mkdir(designsdir)
os.system(f"cp {current_rfd} {designsdir}/")

chains_to_design = "A B C D E F G H"
# MspA target positions to fix sequence
fixed_positions="84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133, 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133"

path_for_parsed_chains=f"./{designsdir}/parsed_pdbs.jsonl"
path_for_tied_positions=f"./{designsdir}/tied_pdbs.jsonl"
path_for_designed_sequences=f"./{designsdir}_mpnn"
path_for_fixed_positions=f"./{designsdir}/fixed_pdbs.jsonl"

rfdbackbones = []
for outpdb in glob.glob(f"{designsdir}/*.pdb"):
  rfdbackbones.append(outpdb)
if len(rfdbackbones) == 0:
  raise Exception("There are no scaffold-guided RFDiffusion backbones to sequence design.")

run_cmd(f"python ProteinMPNN/helper_scripts/parse_multiple_chains.py --input_path={designsdir} --output_path={path_for_parsed_chains}")
run_cmd(f'python ProteinMPNN/helper_scripts/make_fixed_positions_dict.py --input_path={path_for_parsed_chains} --output_path={path_for_fixed_positions} --chain_list "{chains_to_design}" --position_list "{fixed_positions}"')
# Create tied positions jsonl file from homooligomer RFdiffusion outputs
run_cmd(f'python ProteinMPNN/helper_scripts/make_tied_positions_dict.py --input_path={path_for_parsed_chains} --output_path={path_for_tied_positions} --homooligomer 1')
run_cmd(f'python ProteinMPNN/protein_mpnn_run.py \
        --path_to_model_weights "./ProteinMPNN/soluble_model_weights/"\
        --jsonl_path {path_for_parsed_chains} \
        --tied_positions_jsonl {path_for_tied_positions} \
        --fixed_positions_jsonl {path_for_fixed_positions} \
        --out_folder {path_for_designed_sequences} \
        --num_seq_per_target {sequences_per_target} \
        --sampling_temp "0.2" \
        --batch_size 1')

alpha_1 = list("ARNDCQEGHILKMFPSTWYV-")
alpha_3 = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
           'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL','GAP']
aa_1_3 = {a:b for a,b in zip(alpha_1,alpha_3)}

def append_chain_to_pose(p1a,p2a,chain=1,new_chain=True):
  jumpadded = False
  for res in pyrosetta.rosetta.core.pose.get_chain_residues(p2a,chain):
    if not jumpadded:
      p1a.append_residue_by_jump(res, 1, "", "", new_chain)
      jumpadded = True
    else:
      p1a.append_residue_by_bond(res)
  return p1a

def thread_mpnn_seq( pose, binder_seq ):
  rsd_set = pose.residue_type_set_for_pose( pyrosetta.rosetta.core.chemical.FULL_ATOM_t )
  for resi, mut_to in enumerate( binder_seq ):
    resi += 1 # 1 indexing
    if pose.residue(resi).name().split(':')[-1] != 'disulfide':
      name3 = aa_1_3[ mut_to ]
      new_res = pyrosetta.rosetta.core.conformation.ResidueFactory.create_residue( rsd_set.name_map( name3 ) )
      pose.replace_residue( resi, new_res, True )
  return pose

import glob
seqs = []
for fasta in glob.glob(f"{path_for_designed_sequences}/seqs/*.fa"):
  input_pdb_name = ""
  with open(fasta) as f:
    scores = []
    for l in f:
      if l.startswith('>'):
        scores = l.split()
        if input_pdb_name == "": input_pdb_name = scores[0][1:-1]
      elif not scores[1].startswith("score="):
        seqs.append([input_pdb_name, scores, l.strip()])
for seq in seqs:
  idx = seq[1][1].split('=')[-1][0:-1]
  input_pdb = f"{designsdir}/{seq[0]}.pdb"
  #print(f'input {input_pdb}')
  pose = pyrosetta.pose_from_file(input_pdb)
  chainseqs = seq[2].split('/')
  singleseq = seq[2].replace('/','')
  pose = thread_mpnn_seq( pose, singleseq )
  chainlen = len(chainseqs[0])
  chains_pose = Pose()
  residue_indices = rosetta.utility.vector1_unsigned_long()
  for i in range(1,chainlen+1):
    residue_indices.append(i)
  pyrosetta.rosetta.core.pose.pdbslice(chains_pose, pose, residue_indices)
  for i in range(1,len(chainseqs)):
    start = chainlen*i+1
    residue_indices = rosetta.utility.vector1_unsigned_long()
    for j in range(start,start+chainlen):
      residue_indices.append(j)
    chainpose = Pose()
    pyrosetta.rosetta.core.pose.pdbslice(chainpose, pose, residue_indices)
    append_pose_to_pose(chains_pose, chainpose, True)

  if minimize:
    mm = MoveMap()
    mm.set_bb(False)  # Fix backbone
    mm.set_chi(True)  # Allow side chains to move
    mm.set_jump(False)
    scorefxn = create_score_function("ref2015")
    min_mover = MinMover()
    min_mover.movemap(mm)
    min_mover.score_function(scorefxn)
    print(f"minimize {path_for_designed_sequences}/{seq[0]}_{idx}.pdb")
    min_mover.apply(chains_pose)

  print(f"output {path_for_designed_sequences}/{seq[0]}_{idx}.pdb")
  chains_pose.dump_pdb(f"{path_for_designed_sequences}/{seq[0]}_{idx}.pdb")


In [ ]:
#@title **Download and install the AlphaFold3 git repo** (~5-10min)
%%time
if not os.path.isdir("alphafold3"):
  # get specific branch this was tested on in 20260204
  os.system("git clone https://github.com/google-deepmind/alphafold3.git; cd alphafold3; git fetch --all; git reset --hard ebfe70a")

if not os.path.isdir('alphafold3_venv'):
  # build the uv environment for running alphafold3
  run_cmd("export UV_COMPILE_BYTECODE=1;export UV_PROJECT_ENVIRONMENT=/content/alphafold3_venv; uv venv $UV_PROJECT_ENVIRONMENT;export PATH=\"/content/alphafold3_venv/bin:$PATH\"; cd alphafold3; uv sync --all-groups ; uv run build_data")


In [ ]:
#@title **Upload or mount drive for AlphaFold 3 model parameters.** (A very long time unless you use your Google Drive)
#@markdown **Copyright 2024 DeepMind Technologies Limited**
#@markdown
#@markdown AlphaFold 3 source code is licensed under CC BY-NC-SA 4.0. To view a copy of
#@markdown this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/
#@markdown
#@markdown To request access to the AlphaFold 3 model parameters, follow the process set
#@markdown out at https://github.com/google-deepmind/alphafold3. You may only use these
#@markdown if received directly from Google. Use is subject to terms of use available at
#@markdown https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md

import os

#@markdown **We recommend first uploading the model parameters to your Goggle Drive and then mount the drive.**

#@markdown Select to mount your Google Drive (you will need to follow the popup instructions to authenticate)
mount_myDrive = True #@param {type:"boolean"}

#@markdown Enter the path to your AlphaFold 3 models directory in 'My Drive'
path_to_models_dir = "af3_models" #@param {type: "string"}

#@markdown The directory above should contain af3.bin, af3.bin.zst, and the params directory

af3_models_dir = ''
if mount_myDrive:
  mount_path = '/content/drive'
  if not os.path.exists(mount_path):
    from google.colab import drive
    drive.mount(mount_path)
  af3_models_dir = f"{mount_path}/MyDrive/{path_to_models_dir}"
  if os.path.isdir(af3_models_dir) and os.path.exists(f"{af3_models_dir}/params/params_model_1_multimer.npz") and os.path.exists(f"{af3_models_dir}/af3.bin"):
    print(f"Using AlphaFold 3 params from your Google Drive at {af3_models_dir}")
  else:
    raise Exception("Could not find models directory. Is your path correct?")
else:
  from google.colab import files
  if not os.path.isdir("af3_models"):
    os.mkdir("af3_models")
  if not os.path.isdir("af3_models/params"):
    os.mkdir("af3_models/params")
  uploads = files.upload()
  for upload_key in list(uploads.keys()):
    upload_name = upload_key.split('/')[-1].split()[0]
    if not os.path.exists(f"af3_models/af3.bin") and upload_name == "af3.bin":
      print(f"Upload upload_name into af3_models/{upload_name}")
      with open(f"af3_models/{upload_name}", "wb") as out: out.write(uploads[upload_key])
    elif not os.path.exists(f"af3_models/af3.bin.zst") and upload_name == "af3.bin.zst":
      print(f"Upload upload_name into af3_models/{upload_name}")
      with open(f"af3_models/{upload_name}", "wb") as out: out.write(uploads[upload_key])
    elif not os.path.exists(f"af3_models/params/{upload_name}"):
      print(f"Upload upload_name into af3_models/params/{upload_name}")
      with open(f"af3_models/params/{upload_name}", "wb") as out: out.write(uploads[upload_key])
  af3_models_dir = 'af3_models'

af3_database_dir = 'af3_database'
if not os.path.isdir(af3_database_dir):
  # create a placeholder database directory
  run_cmd(f"mkdir {af3_database_dir}; cd {af3_database_dir}; touch bfd-first_non_consensus_sequences.fasta mgy_clusters_2022_05.fa nt_rna_2023_02_23_clust_seq_id_90_cov_80_rep_seq.fasta pdb_seqres_2022_09_28.fasta rfam_14_9_clust_seq_id_90_cov_80_rep_seq.fasta rnacentral_active_seq_id_90_cov_80_linclust.fasta uniprot_all_2021_04.fa uniref90_2022_05.fa; mkdir mmcif_files")
print(af3_models_dir)
run_cmd(f"ls -lh {af3_models_dir}")

In [ ]:
#@title **Run AlphaFold 3 on the MPNN outputs** (5-10min per model, A100 GPU or Nvidia GPU compute capability 8.0+ required)
%%time

# Copyright 2024 DeepMind Technologies Limited
#
# AlphaFold 3 source code is licensed under CC BY-NC-SA 4.0. To view a copy of
# this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/
#
# To request access to the AlphaFold 3 model parameters, follow the process set
# out at https://github.com/google-deepmind/alphafold3. You may only use these
# if received directly from Google. Use is subject to terms of use available at
# https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md

#@markdown This cell will run AlphaFold 3 without using MSAs from a genetic or template search.

#@markdown Alternatively you can download the MPNN sequences and submit them to the AlphaFold 3 server ending here in this notebook.

#@markdown The MPNN results are located in the mspa_rfd_mpnn folder.
import glob
import json

def read_pdb_atom(l):
  chain = l[20:22].strip()
  atype = l[11:17].strip()
  name3 = l[17:20].strip()
  resnum = int(l[22:26].strip())
  x = float(l[30:38])
  y = float(l[38:46])
  z = float(l[46:54])
  return chain, atype, name3, resnum, x, y, z

longer_names = {'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D',
              'CYS': 'C', 'GLU': 'E', 'GLN': 'Q', 'GLY': 'G',
              'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K',
              'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S',
              'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V' }

for mpnnout in glob.glob(f'{path_for_designed_sequences}/*.pdb'):
  in_name = mpnnout.split('/')[-1].split('.pdb')[0]
  chain_seqs = {}
  # get chains and single letter sequences
  with open(mpnnout) as f:
    for l in f:
      if l.startswith("ATOM"):
        chain, atype, name3, resnum, x, y, z = read_pdb_atom(l)
        if atype == 'CA':
          if not chain in chain_seqs:
            chain_seqs[chain] = longer_names[name3]
          else:
            chain_seqs[chain] += longer_names[name3]
  chains = []
  prevseq = ''
  for chain in chain_seqs:
    chains.append(chain)
    if prevseq and chain_seqs[chain] != prevseq:
      raise Exception(f"{mpnnout} is not a homo-oligomer.")
    prevseq = chain_seqs[chain]

  # Create AF3 input json file
  json_in = {
    "name": in_name,
    "sequences": [
      {
        "protein": {
          "id": chains,
          "sequence": prevseq,
          "unpairedMsa": "",
          "pairedMsa": "",
          "templates": ""
        }
      }
    ],
    "modelSeeds": [1],
    "dialect": "alphafold3",
    "version": 1
  }
  with open(f"{path_for_designed_sequences}/{in_name}.json", "w") as json_file:
    json.dump(json_in, json_file, indent=4)

# Run AF3 on each input json file
for json_in in glob.glob(f"{path_for_designed_sequences}/*.json"):
  name = json_in.split('/')[-1].split('.json')[0]
  json_prefix = json_in.split('.json')[0]
  print(name)
  cmd = f"export JAX_PLATFORMS=cuda; alphafold3_venv/bin/python "
  cmd += f"alphafold3/run_alphafold.py --json_path={json_in} "
  cmd+= f"--db_dir={af3_database_dir} --model_dir={af3_models_dir} "
  cmd+= f"--output_dir=af3_output > {json_prefix}.run_log 2>&1"
  print(cmd)
  run_cmd(cmd)


In [ ]:
#@title **Process AlphaFold 3 results**
%%time

import json
import pandas as pd
import glob
import os
import statistics
from Bio.PDB import MMCIFParser, PDBIO

import pyrosetta
from pyrosetta.rosetta import *
from pyrosetta.rosetta.core import *
pyrosetta.init(" -mute all ")

af3_mods = []
output_csv_path = f'af3_predictions.csv'
parsed_df = pd.DataFrame()
if os.path.exists(output_csv_path):
  parsed_df = pd.read_csv(output_csv_path, header=0)
else:
  parsed_data = []
  output_dir_list = glob.glob(f'af3_output/*')
  for output_dir in output_dir_list:
    for filename in os.listdir(output_dir):
      if filename.endswith('summary_confidences.json'):
        file_path = os.path.join(output_dir, filename)
        with open(file_path, 'r') as file:
          data = json.load(file)
          file_path2= file_path.split('summary_')[0] + file_path.split('summary_')[-1]
          with open(file_path2, 'r') as file2:
            data2 = json.load(file2)
          # Extract the required values
          iptm = data.get("iptm")
          ptm = data.get("ptm")
          # Extract the PAE values
          pae_values = data["chain_pair_pae_min"]
          plddt_total = statistics.mean(data2["atom_plddts"])
          design_name = output_dir.split('/')[-1]

          af3_model = f'{output_dir}/{design_name}_model.cif'
          mpnn_model = f'{path_for_designed_sequences}/{design_name}.pdb'

          af3_model_pdb = af3_model.split('.cif')[0]+'_af3.pdb'
          if not os.path.exists(af3_model_pdb):
            parser = MMCIFParser()
            structure = parser.get_structure(design_name, af3_model)
            io = PDBIO()
            io.set_structure(structure)
            io.save(af3_model_pdb)

          # superimpose mpnn_model to af3_model
          af3_pose = pose_from_file(af3_model_pdb)
          mpnn_pose = pose_from_file(mpnn_model)
          af3_chains = af3_pose.split_by_chain()
          mpnn_chains = mpnn_pose.split_by_chain()
          rmsd = scoring.CA_rmsd(af3_pose, mpnn_pose, 1, af3_pose.size())
          rmsd_subunit = scoring.CA_rmsd(af3_chains[1], mpnn_chains[1], 1, af3_chains[1].size())
          parsed_entry = {
            'design_name': design_name,
            'af3_model': af3_model_pdb,
            'mpnn_model': mpnn_model,
            'iptm': iptm,
            'ptm': ptm,
            'plddt': plddt_total,
            'rmsd': rmsd,
            'rmsd_subunit': rmsd_subunit
          }
          #print(f"{af3_model_pdb} iptm: {iptm:.2f} ptm: {ptm:.2f} plddt: {plddt_total:.2f} rmsd: {rmsd:.2f} rmsd_subunit: {rmsd_subunit:.2f}")
          # Append the parsed entry to the list
          parsed_data.append(parsed_entry)
          # add score info to af3 pdb
          has_scores = False
          fstr = ''
          with open(af3_model_pdb) as f:
            for l in f:
              if l.startswith("REMARK   af3_scores"):
                has_scores = True
                break
              else:
                fstr += l
          if not has_scores:
            nf = open(f"{af3_model_pdb}.tmp", 'w')
            nf.write(f"REMARK   af3_scores iptm: {iptm:.2f} ptm: {ptm:.2f} plddt: {plddt_total:.2f} rmsd: {rmsd:.2f} rmsd_subunit: {rmsd_subunit:.2f}\n")
            nf.write(fstr)
            nf.close()
            os.system(f"mv {af3_model_pdb}.tmp {af3_model_pdb}")
          af3_mods.append(af3_model_pdb)

  # Convert the list to a DataFrame
  parsed_df = pd.DataFrame(parsed_data)
  # Save the parsed DataFrame to a CSV file
  #parsed_df.to_csv(output_csv_path, index=False)

parsed_df

In [ ]:
#@title **Select AlphaFold 3 model to download**
import py3Dmol
import ipywidgets as widgets
from ipywidgets import interact, Layout
from IPython.display import display, clear_output

current_af3 = ""
af3_mods = { 'Select model to download': ''}
for i,r in parsed_df.iterrows():
  af3_mods[f"{r['af3_model']}  iptm: {r['iptm']:.2f} ptm: {r['ptm']:.2f} plddt: {r['plddt']:.2f} rmsd: {r['rmsd']:.2f} rmsd_subunit: {r['rmsd_subunit']:.2f}"] = r['af3_model']

dropdown = widgets.Dropdown(
  options=af3_mods,
  description='af2 wrap:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(af3_mod):
  if os.path.exists(af3_mod):
    print(af3_mod)
    global current_af3
    current_af3 = af3_mod
    clear_output(wait=True)
    print()
    print(current_af3)
    view = py3Dmol.view(width=500, height=400)
    with open(af3_mod, "r") as f:
      pdb_data = f.read()
    view.addModel(pdb_data, 'pdb')
    view.setStyle({'cartoon': {'colorscheme': 'chain'}})
    view.zoomTo()
    view.show()

widgets.interact(on_dropdown_change, af3_mod=dropdown);

In [ ]:
#@title **Download selected AlphaFold 3 model**
from google.colab import files
files.download(current_af3)